#**A05 Multilayer Perceptron**

## Julieta Madrigal Flores - 744029
### Jueves 09 abril 2026

In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import warnings #  suprime los mensajes de advertencia para que la salida sea más limpia = es que me salían demasiados
warnings.filterwarnings('ignore')

In [ ]:
# Semilla para reproducibilidad
np.random.seed(42)

# CARGAR DATASET
data = pd.read_csv('Heart Prediction Quantum Dataset.csv')

In [ ]:
# Preparar X y Y
Y = data['HeartDisease']
X = data.drop(['HeartDisease', 'QuantumPatternFeature'], axis=1)

In [ ]:
# Definir columnas
X_numericas = ['Age', 'BloodPressure', 'Cholesterol', 'HeartRate']
X_categoricas = ['Gender']

In [ ]:
# Definir el escalador
num_transformer = StandardScaler()

In [ ]:
# Definir el preprocesador
preprocessor = ColumnTransformer(transformers=[('num', num_transformer, X_numericas)])

## INTENTO 1

In [ ]:
# Definir la función objetivo
# Recibe una lista de 3 valores = las 3 capas
def f(params):

    # 3 valores recibidos, osea uno a cada capa
    capa1, capa2, capa3 = params

    # Crear pipeline con la arquitectura específica
    # crea la red neuronal con las 3 capas ocultas y convierte los valores a enteros
    # a pesar de que el Proceso Gaussiano trabaja con decimales, las neuronas deben ser números enteros
    model = MLPClassifier(hidden_layer_sizes=(int(capa1), int(capa2), int(capa3)),max_iter=500,random_state=42)
    pipeline = Pipeline(steps=[('procesador', preprocessor),('modelo', model)])

    # Calcular cross-validation score
    # validación cruzada de 10 folds
    # mezclando los datos antes de dividir
    # usando ROC-AUC como métrico
    scores = cross_val_score(pipeline, X, Y,scoring='roc_auc_ovr',cv=StratifiedKFold(10, shuffle=True, random_state=42))

    return np.mean(scores)

# Definir rangos
capa1_min, capa1_max = 2, 6
capa2_min, capa2_max = 5, 15
capa3_min, capa3_max = 3, 6

# DATOS INICIALES (3 puntos aleatorios)
# construye 3 configuraciones aleatorias dentro de los rangos definidos para arrancar el algoritmo
X_train = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(3, 3))
y_train = np.array([f(params) for params in X_train])

# Guardar puntos base
X_base = X_train.copy()
y_base = y_train.copy()

# Listas para puntos agregados
puntos_agregados_x = []
puntos_agregados_y = []

# 30 iteraciones
for r in range(1, 31):

    # Entrenar GP con los datos actuales
    modelo = GaussianProcessRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    # Crear 1000 puntos candidatos
    X_test = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(1000, 3))

    # Predecir
    y_pred_mean, y_pred_std = modelo.predict(X_test, return_std=True)

    # Límite superior (media + 2*sigma) - MAXIMIZAR
    limite_superior = y_pred_mean + 2 * y_pred_std

    # Encontrar punto con límite superior máximo
    # encuentra el índice del candidato con el límite superior más alto
    idx_max = np.argmax(limite_superior)

    # extrae esa combinación ganadora del conjunto de candidatos
    params_nuevos = X_test[idx_max]

    # Evaluar función real en ese punto
    # y la evalúa con la red neuronal real para obtener su score verdadero
    score_nuevo = f(params_nuevos)

    # Guardar
    puntos_agregados_x.append(params_nuevos)
    puntos_agregados_y.append(score_nuevo)

    # Mostrar progreso
    mejor_score_actual = np.max(y_train)
    if score_nuevo > mejor_score_actual:
      # si el score supera al mejor hasta ahora, agrega una marca visual en la impresión
        marca = " NUEVO MEJOR!"
    else:
        marca = ""

  # imprime en que regresión va, su score y el mejor score hsasta el momento
    print(f"Regresión {r:2d}: Capas=[{params_nuevos[0]:.0f}, {params_nuevos[1]:.0f}, {params_nuevos[2]:.0f}] | "
          f"Score={score_nuevo:.4f} | Mejor hasta ahora={mejor_score_actual:.4f} {marca}")

    # Agregar punto para siguiente iteración
    X_train = np.vstack([X_train, params_nuevos])
    y_train = np.append(y_train, score_nuevo)

# Encontrar la mejor combinación
idx_mejor = np.argmax(y_train)
# extrae la combinación que logró ese mejor score
mejor_params = X_train[idx_mejor]
mejor_score = y_train[idx_mejor]

print(f"\n MEJOR CONFIGURACIÓN ENCONTRADA:")
# imprime el resultado final con la mejor combinación y su score
print(f"   Capa 1: {mejor_params[0]:.0f} neuronas (rango {capa1_min}-{capa1_max})")
print(f"   Capa 2: {mejor_params[1]:.0f} neuronas (rango {capa2_min}-{capa2_max})")
print(f"   Capa 3: {mejor_params[2]:.0f} neuronas (rango {capa3_min}-{capa3_max})")
print(f"\n MEJOR SCORE (ROC-AUC media con CV=10): {mejor_score:.4f}")


Regresión  1: Capas=[2, 15, 5] | Score=0.8229 | Mejor hasta ahora=0.8461 
Regresión  2: Capas=[3, 14, 4] | Score=0.8121 | Mejor hasta ahora=0.8461 
Regresión  3: Capas=[4, 13, 5] | Score=0.8285 | Mejor hasta ahora=0.8461 
Regresión  4: Capas=[5, 14, 4] | Score=0.8380 | Mejor hasta ahora=0.8461 
Regresión  5: Capas=[4, 13, 4] | Score=0.8138 | Mejor hasta ahora=0.8461 
Regresión  6: Capas=[5, 12, 5] | Score=0.8497 | Mejor hasta ahora=0.8461  NUEVO MEJOR!
Regresión  7: Capas=[5, 14, 6] | Score=0.8187 | Mejor hasta ahora=0.8497 
Regresión  8: Capas=[6, 7, 4] | Score=0.7948 | Mejor hasta ahora=0.8497 
Regresión  9: Capas=[5, 8, 3] | Score=0.8317 | Mejor hasta ahora=0.8497 
Regresión 10: Capas=[5, 7, 4] | Score=0.8400 | Mejor hasta ahora=0.8497 
Regresión 11: Capas=[3, 8, 4] | Score=0.8149 | Mejor hasta ahora=0.8497 
Regresión 12: Capas=[3, 7, 5] | Score=0.8236 | Mejor hasta ahora=0.8497 
Regresión 13: Capas=[4, 9, 5] | Score=0.8352 | Mejor hasta ahora=0.8497 
Regresión 14: Capas=[2, 9, 5] |

## INTENTO 2

Todo es igual excepto por n_initial = 5 sube los puntos iniciales de 3 a 5, dándole al Proceso Gaussiano más información desde el arranque. n_iteraciones = 50 aumenta el número de iteraciones de 30 a 50 para explorar más y n_candidatos = 2000 duplica los candidatos evaluados por vuelta de 1000 a 2000, lo que hace que el Proceso Gaussiano tenga más opciones para elegir la mejor siguiente configuración.


Este intento solo se muestra el top 10 de mejores configuraciones encontradas ordenadas de mayor a menor score.

In [ ]:
# Definir la función objetivo
def f(params):
    capa1, capa2, capa3 = params

    # Crear pipeline con la arquitectura específica
    model = MLPClassifier(hidden_layer_sizes=(int(capa1), int(capa2), int(capa3)),max_iter=500,random_state=42)

    pipeline = Pipeline(steps=[('procesador', preprocessor),('modelo', model)])

    # Calcular cross-validation score
    scores = cross_val_score(pipeline, X, Y,scoring='roc_auc_ovr',cv=StratifiedKFold(10, shuffle=True, random_state=42))
    return np.mean(scores)

# Definir rangos
capa1_min, capa1_max = 2, 6
capa2_min, capa2_max = 5, 15
capa3_min, capa3_max = 3, 6

# DATOS INICIALES
n_initial = 5
X_train = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_initial, 3))
y_train = np.array([f(params) for params in X_train])

# Guardar puntos base
X_base = X_train.copy()
y_base = y_train.copy()

# Listas para puntos agregados
puntos_agregados_x = []
puntos_agregados_y = []

# 50 iteraciones
n_iteraciones = 50
# le sumamos uno más para que haga 50, sino haría 49
for r in range(1, n_iteraciones + 1):

    # Entrenar GP con los datos actuales
    modelo = GaussianProcessRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    # Crear puntos candidatos = son los exploradores
    n_candidatos = 2000
    X_test = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_candidatos, 3))

    # Predecir
    y_pred_mean, y_pred_std = modelo.predict(X_test, return_std=True)

    # Límite superior (media + 2*sigma) - MAXIMIZAR
    limite_superior = y_pred_mean + 2 * y_pred_std

    # Encontrar punto con límite superior máximo
    idx_max = np.argmax(limite_superior)
    params_nuevos = X_test[idx_max]

    # Evaluar función real en ese punto
    score_nuevo = f(params_nuevos)

    # Guardar
    puntos_agregados_x.append(params_nuevos)
    puntos_agregados_y.append(score_nuevo)

    # Agregar punto para siguiente iteración
    X_train = np.vstack([X_train, params_nuevos])
    y_train = np.append(y_train, score_nuevo)

# Mostrar top 10 mejores configuraciones
print("\n TOP 10 MEJORES CONFIGURACIONES:")
indices_ordenados = np.argsort(y_train)[::-1]
for i in range(min(10, len(indices_ordenados))):
    idx = indices_ordenados[i]
    config = X_train[idx]
    score = y_train[idx]
    ganadora = " GANADORA" if i == 0 else ""
    print(f"   {i+1:2d}. [{config[0]:.0f}, {config[1]:.0f}, {config[2]:.0f}] -> Score: {score:.4f} {ganadora}")


 TOP 10 MEJORES CONFIGURACIONES:
    1. [6, 11, 3] -> Score: 0.8577  GANADORA
    2. [6, 14, 4] -> Score: 0.8557 
    3. [6, 13, 6] -> Score: 0.8527 
    4. [6, 12, 4] -> Score: 0.8497 
    5. [5, 9, 4] -> Score: 0.8487 
    6. [3, 13, 6] -> Score: 0.8478 
    7. [2, 12, 5] -> Score: 0.8474 
    8. [2, 14, 5] -> Score: 0.8461 
    9. [6, 15, 5] -> Score: 0.8455 
   10. [6, 10, 6] -> Score: 0.8453 


## INTENTO 3

Mantiene los mismos 5 puntos iniciales y 2000 candidatos del intento 2, pero sube las iteraciones a 100 y muestra el top 10 de las combinaciones.

In [ ]:
# Definir la función objetivo
def f(params):
    capa1, capa2, capa3 = params

    # Crear pipeline con la arquitectura específica
    model = MLPClassifier(hidden_layer_sizes=(int(capa1), int(capa2), int(capa3)),max_iter=500,random_state=42)
    pipeline = Pipeline(steps=[('procesador', preprocessor),('modelo', model)])

    # Calcular cross-validation score
    scores = cross_val_score(pipeline, X, Y,scoring='roc_auc_ovr',cv=StratifiedKFold(10, shuffle=True, random_state=42))
    return np.mean(scores)

# Definir rangos
capa1_min, capa1_max = 2, 6
capa2_min, capa2_max = 5, 15
capa3_min, capa3_max = 3, 6

# DATOS INICIALES
n_initial = 5
X_train = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_initial, 3))
y_train = np.array([f(params) for params in X_train])

# Guardar puntos base
X_base = X_train.copy()
y_base = y_train.copy()

# Listas para puntos agregados
puntos_agregados_x = []
puntos_agregados_y = []

# 100 iteraciones
n_iteraciones = 100
mejores_por_iteracion = [np.max(y_train)]

for r in range(1, n_iteraciones + 1):

    # Entrenar GP con los datos actuales
    modelo = GaussianProcessRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    # Crear puntos candidatos
    n_candidatos = 2000
    X_test = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_candidatos, 3))

    # Predecir
    y_pred_mean, y_pred_std = modelo.predict(X_test, return_std=True)

    # Límite superior (media + 2*sigma) - MAXIMIZAR
    limite_superior = y_pred_mean + 2 * y_pred_std

    # Encontrar punto con límite superior máximo
    idx_max = np.argmax(limite_superior)
    params_nuevos = X_test[idx_max]

    # Evaluar función real en ese punto
    score_nuevo = f(params_nuevos)

    # Guardar
    puntos_agregados_x.append(params_nuevos)
    puntos_agregados_y.append(score_nuevo)

    # Agregar punto para siguiente iteración
    X_train = np.vstack([X_train, params_nuevos])
    y_train = np.append(y_train, score_nuevo)

# Mostrar top 10 mejores configuraciones
print("\n TOP 10 MEJORES CONFIGURACIONES:")
indices_ordenados = np.argsort(y_train)[::-1]
for i in range(min(10, len(indices_ordenados))):
    idx = indices_ordenados[i]
    config = X_train[idx]
    score = y_train[idx]
    ganadora = " GANADORA" if i == 0 else ""
    print(f"   {i+1:2d}. [{config[0]:.0f}, {config[1]:.0f}, {config[2]:.0f}] -> Score: {score:.4f} {ganadora}")


 TOP 10 MEJORES CONFIGURACIONES:
    1. [5, 12, 5] -> Score: 0.8578  GANADORA
    2. [6, 12, 6] -> Score: 0.8578 
    3. [5, 11, 4] -> Score: 0.8577 
    4. [6, 12, 3] -> Score: 0.8577 
    5. [6, 11, 6] -> Score: 0.8561 
    6. [6, 14, 4] -> Score: 0.8557 
    7. [5, 7, 5] -> Score: 0.8530 
    8. [6, 14, 5] -> Score: 0.8527 
    9. [6, 15, 4] -> Score: 0.8510 
   10. [6, 13, 3] -> Score: 0.8503 


## INTENTO 4

n_initial  sube los puntos iniciales a 15, n_iteraciones  a 200 y n_candidatos = 5000; además, se muestra el top 15 = más que nada solo se amplía todo para cubrir el espacio de búsqueda de la forma más exhaustiva posible.

In [ ]:
# Definir la función objetivo
def f(params):
    capa1, capa2, capa3 = params

    model = MLPClassifier(hidden_layer_sizes=(int(capa1), int(capa2), int(capa3)),max_iter=500,random_state=42)
    pipeline = Pipeline(steps=[('procesador', preprocessor),('modelo', model)])
    scores = cross_val_score(pipeline, X, Y,scoring='roc_auc_ovr',cv=StratifiedKFold(10, shuffle=True, random_state=42))

    return np.mean(scores)

# Definir rangos
capa1_min, capa1_max = 2, 6
capa2_min, capa2_max = 5, 15
capa3_min, capa3_max = 3, 6

n_initial = 15  # puntos iniciales
n_iteraciones = 200  # iteraciones
n_candidatos = 5000  # candidatos por iteración

# DATOS INICIALES
X_train = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_initial, 3))
y_train = np.array([f(params) for params in X_train])

# Guardar puntos base
X_base = X_train.copy()
y_base = y_train.copy()

# Listas para puntos agregados
puntos_agregados_x = []
puntos_agregados_y = []

mejor_inicial = np.max(y_train)

# iteraciones
mejores_por_iteracion = [mejor_inicial]
ultimo_record = 0

for r in range(1, n_iteraciones + 1):

    # Entrenar GP con los datos actuales
    modelo = GaussianProcessRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    # Crear puntos candidatos
    X_test = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_candidatos, 3))

    # Predecir
    y_pred_mean, y_pred_std = modelo.predict(X_test, return_std=True)

    # Límite superior (media + 2*sigma) - MAXIMIZAR
    limite_superior = y_pred_mean + 2 * y_pred_std

    # Encontrar punto con límite superior máximo
    idx_max = np.argmax(limite_superior)
    params_nuevos = X_test[idx_max]

    # Evaluar función real en ese punto
    score_nuevo = f(params_nuevos)

    # Guardar
    puntos_agregados_x.append(params_nuevos)
    puntos_agregados_y.append(score_nuevo)

    # Agregar punto para siguiente iteración
    X_train = np.vstack([X_train, params_nuevos])
    y_train = np.append(y_train, score_nuevo)

# Mostrar top 15 mejores configuraciones
print("\n TOP 15 MEJORES CONFIGURACIONES:")
indices_ordenados = np.argsort(y_train)[::-1]
for i in range(min(15, len(indices_ordenados))):
    idx = indices_ordenados[i]
    config = X_train[idx]
    score = y_train[idx]
    ganadora = " GANADORA" if i == 0 else ""
    print(f"   {i+1:2d}. [{config[0]:.0f}, {config[1]:.0f}, {config[2]:.0f}] -> Score: {score:.4f} {ganadora}")



 TOP 15 MEJORES CONFIGURACIONES:
    1. [6, 12, 5] -> Score: 0.8578  GANADORA
    2. [5, 12, 5] -> Score: 0.8578 
    3. [6, 12, 6] -> Score: 0.8578 
    4. [5, 13, 6] -> Score: 0.8578 
    5. [6, 12, 3] -> Score: 0.8577 
    6. [6, 11, 4] -> Score: 0.8577 
    7. [6, 10, 6] -> Score: 0.8561 
    8. [5, 11, 6] -> Score: 0.8561 
    9. [6, 13, 5] -> Score: 0.8557 
   10. [5, 6, 6] -> Score: 0.8530 
   11. [6, 7, 6] -> Score: 0.8530 
   12. [6, 13, 5] -> Score: 0.8527 
   13. [5, 14, 5] -> Score: 0.8527 
   14. [6, 13, 6] -> Score: 0.8527 
   15. [6, 15, 4] -> Score: 0.8510 


## INTENTO 5

Se mantiene todo igual, solo que al final imprime únicamente la combinación cuando se rompe un récord.

In [ ]:
# FUNCIÓN OBJETIVO
def f(params):
    capa1, capa2, capa3 = params
    model = MLPClassifier(hidden_layer_sizes=(int(capa1), int(capa2), int(capa3)),max_iter=500,random_state=42)
    pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

    # Con shuffle para mejor distribución de clases
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

    return np.mean(scores)

# RANGOS DE LAS 3 CAPAS
capa1_min, capa1_max = 2, 6
capa2_min, capa2_max = 5, 15
capa3_min, capa3_max = 3, 6

# CONFIGURACIÓN
n_initial = 5
n_iteraciones = 50
n_candidatos = 2000

# PUNTOS INICIALES
X_train = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_initial, 3))
y_train = np.array([f(params) for params in X_train])

# BUCLE PRINCIPAL
for r in range(1, n_iteraciones + 1):
    modelo = GaussianProcessRegressor(random_state=42)
    modelo.fit(X_train, y_train)

    X_test = np.random.uniform([capa1_min, capa2_min, capa3_min],[capa1_max, capa2_max, capa3_max],(n_candidatos, 3))

    y_pred_mean, y_pred_std = modelo.predict(X_test, return_std=True)
    limite_superior = y_pred_mean + 2 * y_pred_std
    idx_max = np.argmax(limite_superior)
    params_nuevos = X_test[idx_max]
    score_nuevo = f(params_nuevos)

    if score_nuevo > np.max(y_train):
        print(f"Iter {r:3d}: [{params_nuevos[0]:.0f}, {params_nuevos[1]:.0f}, {params_nuevos[2]:.0f}] -> Score: {score_nuevo:.4f}")

    X_train = np.vstack([X_train, params_nuevos])
    y_train = np.append(y_train, score_nuevo)

# RESULTADO FINAL
idx_mejor = np.argmax(y_train)
mejor_params = X_train[idx_mejor]
mejor_score = y_train[idx_mejor]

print(f" MEJOR CONFIGURACIÓN:")
print(f"   Capa 1: {mejor_params[0]:.0f} neuronas")
print(f"   Capa 2: {mejor_params[1]:.0f} neuronas")
print(f"   Capa 3: {mejor_params[2]:.0f} neuronas")
print(f"\n SCORE: {mejor_score:.4f}")

Iter   1: [5, 10, 6] -> Score: 0.8561
Iter  17: [6, 11, 4] -> Score: 0.8577
 MEJOR CONFIGURACIÓN:
   Capa 1: 6 neuronas
   Capa 2: 11 neuronas
   Capa 3: 4 neuronas

 SCORE: 0.8577


## COMBINACIONES ENCONTRADAS

In [ ]:
capa1, capa2, capa3 =6,11,4

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

# muestra el score de los 10 folds
print(f"Array de scores: {scores}")
# el promedio de esos 10 folds
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.85833333 0.78666667 0.77583333 0.87583333 0.86666667 0.865
 0.81666667 0.88666667 0.83       0.9       ]
Promedio: 0.8462


In [ ]:
capa1, capa2, capa3 =6,11,3

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

print(f"Array de scores: {scores}")
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.79083333 0.75666667 0.785      0.7925     0.88666667 0.825
 0.76       0.89333333 0.82583333 0.86583333]
Promedio: 0.8182


In [ ]:
capa1, capa2, capa3 =5,12,5

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

print(f"Array de scores: {scores}")
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.88       0.8        0.81333333 0.81333333 0.89666667 0.84166667
 0.84166667 0.94333333 0.84833333 0.9       ]
Promedio: 0.8578


In [ ]:
capa1, capa2, capa3 =6,12,6

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

print(f"Array de scores: {scores}")
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.87       0.78666667 0.815      0.82666667 0.89166667 0.84833333
 0.785      0.885      0.805      0.86      ]
Promedio: 0.8373


In [ ]:
capa1, capa2, capa3 =6,12,5

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

print(f"Array de scores: {scores}")
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.86833333 0.7825     0.805      0.72916667 0.90416667 0.825
 0.72833333 0.79666667 0.785      0.88      ]
Promedio: 0.8104


##**MEJOR COMBINACIÓN = [5 ,13 ,6] = 0.8598**

In [ ]:
capa1, capa2, capa3 =5,13,6

model = MLPClassifier(hidden_layer_sizes=(capa1, capa2, capa3),max_iter=500,random_state=42)
pipeline = Pipeline(steps=[('procesador', preprocessor), ('modelo', model)])

# Evaluar
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, Y, scoring='roc_auc_ovr', cv=cv)

print(f"Array de scores: {scores}")
print(f"Promedio: {np.mean(scores):.4f}")

Array de scores: [0.88       0.85333333 0.81166667 0.80833333 0.91833333 0.83
 0.83333333 0.91833333 0.82833333 0.91666667]
Promedio: 0.8598


#Conclusión

La mejor combinación encontrada fue [5, 13, 6] con un ROC-AUC promedio de 0.8598, esto significa que el modelo es capaz de distinguir correctamente entre pacientes con y sin enfermedad del corazón en aproximadamente 86 de cada 100 casos. Ademas, en los cinco intentos, el algoritmo de Optimización Bayesiana no lograba superar el score promedio de 0.8577-0.8578, sin importar cuántos puntos iniciales, iteraciones o candidatos se agregaran. Esto indica que el algoritmo ya había explorado bien el espacio y que ese era el techo natural dentro de los rangos definidos y ya no hay espacio para mejorar el score.

Para mejorar tendríamos que ampliar el rango de neuronas, agregar más capas o ajustar otros hiperparámetros del modelo. Si se volviera a correr el código los resultados podrían variar un poco porque el shuffle=True del StratifiedKFold mezcla los datos antes de dividirlos y aunque siempre se forman 10 folds, no siempre contienen exactamente los mismos registros. Aunque esto es bueno para evitar sesgos por posibles órdenes implícitos en los datos, el score no es perfectamente reproducible; esto se puede observar en la mejor combinación porque al en el intento, obtuvo un score de 0.8578 pero cuando se volvío a ejecutar arrojó un score de 0.8598.